In [1]:
import pandas as pd
import numpy as np
import re

### Filterung der Datei

In [ ]:
data = pd.read_csv(r"C:/Users/QJ095K/Desktop/2026-03-10 Liste Faltschachteln.csv", sep=';',header=0,encoding='ISO-8859-1')
df = pd.DataFrame(data=data)
#Leere Spalten dropen
df_clean=df.dropna(axis=1,how='all')
df_clean.drop(columns=["Spalte6","Spalte7"])
#Materialnummer aus Object ID extrahieren, um SAP Eingabe zu ermöglichen
df_clean["materialNumber"] = (
    df_clean["Object ID"]
    .astype(str)
    .str.extract(r"^0*([0-9]+)(?:/00)?$")[0]
)
#nur die wichtigen Spalten beibehalten
df_min = df_clean[['Object Name',"materialNumber","Object ID","Length in mm internal","Width in mm internal","Height in mm internal","Amount per bin box (LHM C)","Status"]]
#Maße aus dem ObjectName extrahieren, da hier die innenmaße angegeben sind
pattern = r'(\d+)\s*[Xx*]\s*(\d+)\s*[Xx*]\s*(\d+)'
df_min[['length', 'width', 'height']] = df_min['Object Name'].str.extract(pattern)

df_min['Stabilität'] = (
    df_min['Object Name']
    .str.extract(r'\b(1E|1B)\b', expand=False)
    .fillna('keine')
)
#inaktive Löschen
df_min = df_min[df_min['Status'] != '99 - Inactive']
#Für die Verpackungen, bei denen die Maße nicht extrahiert werden konnten, die Standardmaße aus der Tabelle verwenden
mask = df_min['length'].isna()
source_cols = ["Length in mm internal",
               "Width in mm internal",
               "Height in mm internal"]
target_cols = ['length', 'width', 'height']
df_min.loc[mask, target_cols] = df_min.loc[mask, source_cols].values
df_min[df_min['length'].isna()].shape[0]
df_clean = df_min.dropna(subset=['height'])
spalten = ['length', 'height', 'width']

#Punkte entfernen und Werte in Text umwandeln
for col in spalten:
    df_clean[col] = df_clean[col].astype(str).str.replace('.', '', regex=False)

#Werte in Zahlen umwandeln
df_clean['length']= pd.to_numeric(df_clean['length'])
df_clean['height']= pd.to_numeric(df_clean['height'])
df_clean['width']= pd.to_numeric(df_clean['width'])
#Ausschließen von Verpackungsmaterialie die nur 2-dimensional sind
df_clean=df_clean[df_clean['height']!=0]
#Nur die relevanten Spalten
Verpackungen= df_clean[['Object Name','length','height','width','materialNumber',"Amount per bin box (LHM C)","Stabilität"]]
#materialNumber als Index festlegen und in string umwandeln
Verpackungen.set_index("materialNumber")
Verpackungen["materialNumber"] = Verpackungen["materialNumber"].astype(str)

### Dokument erstellen

In [9]:
Verpackungen.to_csv('meine_datei.csv',sep=',', index=False, encoding='utf-8')